In [ ]:
# %pip install flask openai-whisper sentence-transformers keybert transformers torch torchvision scipy opencv-python Pillow


In [ ]:
import json, os, uuid, threading, subprocess, traceback, re, time, warnings, accelerate
from pathlib import Path
from flask import Flask, render_template, request, jsonify, url_for, redirect

warnings.filterwarnings("ignore")
os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["HF_DATASETS_DISABLE_TORCHCODEC"] = "1"
os.environ["HF_DATASETS_AUDIO_BACKEND"] = "soundfile"

# ═══════════════════════════════════════════════════════════════════
#  Configuration
# ═══════════════════════════════════════════════════════════════════
app = Flask(__name__)
app.config["MAX_CONTENT_LENGTH"] = 2 * 1024 * 1024 * 1024   # 2 GB

UPLOAD_FOLDER     = os.path.join("static", "videos")
TRANSCRIPT_FOLDER = os.path.join("static", "transcripts")
OUTPUT_FOLDER     = "Output"
CHAPTER_DIR       = os.path.join(OUTPUT_FOLDER, "tumbling_window")   # chapters live here
ALLOWED_EXT       = {".mp4", ".webm", ".ogg", ".mkv", ".mov"}
FRAME_SAMPLE_RATE = 60   # Extract 1 frame every N seconds for visual features
MAX_CHAPTER_WORDS = 300  # Split chapters that exceed this word count

os.makedirs(UPLOAD_FOLDER, exist_ok=True)
os.makedirs(TRANSCRIPT_FOLDER, exist_ok=True)
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
os.makedirs(CHAPTER_DIR, exist_ok=True)

# In-memory job tracker  {job_id: {status, step, progress, error, logs}}
jobs = {}

LOW_VALUE_KEYWORDS = [
    "introduction", "intro", "summary", "conclusion", "setup", "support",
    "student feedback", "welcome", "housekeeping", "announcements", "admin",
    "example", "installation", "environment setup", "recap", "review",
    "demo", "q&a", "off-topic", "discussion", "lab", "policies",
    "course overview", "tutorial", "basics", "overview", "tips",
    "preparation", "assignment", "project", "group formation",
    "lab allocation", "examples",
]


# ═══════════════════════════════════════════════════════════════════
#  Helper utilities
# ═══════════════════════════════════════════════════════════════════
def parse_time(t: str) -> int:
    parts = t.split(":")
    if len(parts) == 3:
        h, m, s = parts
        return int(h) * 3600 + int(m) * 60 + int(float(s))
    if len(parts) == 2:
        m, s = parts
        return int(m) * 60 + int(float(s))
    return 0


def fmt_ts(seconds: float) -> str:
    s = int(seconds)
    return f"{s // 3600:02d}:{(s % 3600) // 60:02d}:{s % 60:02d}"


def is_low_value(ch: dict) -> bool:
    text = (ch.get("chapter", "") + " " + ch.get("description", "")).lower()
    return any(k in text for k in LOW_VALUE_KEYWORDS)


def enrich_chapters(chapters: list) -> list:
    enriched = []
    for ch in chapters:
        s = parse_time(ch["start_time"])
        e = parse_time(ch["end_time"])
        # Prefer the LLM-classified "key" field; fall back to keyword heuristic
        key_str = ch.get("key", "")
        if key_str in ("True", "False"):
            low_value = (key_str == "False")
        else:
            low_value = is_low_value(ch)
        enriched.append({**ch, "start_sec": s, "end_sec": e, "low_value": low_value})
    return enriched


def find_chapter_file(stem: str, chapter_dir: str):
    """Find a chapter JSON file for the given video stem in chapter_dir.

    Tries the exact stem with common naming suffixes, then retries after
    stripping a trailing '-full' from the stem to match older files that
    were generated from truncated video names.
    """
    if not os.path.isdir(chapter_dir):
        return None

    # Candidate stems: original, then with trailing '-full' stripped
    stems = [stem]
    if stem.endswith("-full"):
        stems.append(stem[:-5])

    suffixes = [
        "-chapters.json",
        "-tm-chapters.json",
        "-full-chapters.json",
        "-full-full-chapters.json",
    ]

    for s in stems:
        for suffix in suffixes:
            path = os.path.join(chapter_dir, s + suffix)
            if os.path.exists(path):
                return path

    return None


def get_available_videos() -> list:
    videos = []
    for f in Path(UPLOAD_FOLDER).iterdir():
        if f.suffix.lower() in ALLOWED_EXT:
            stem = f.stem
            has_ch = find_chapter_file(stem, CHAPTER_DIR) is not None
            videos.append({"name": f.name, "stem": stem, "has_chapters": has_ch})
    return sorted(videos, key=lambda v: v["name"])


def load_chapters(video_name: str) -> list:
    stem = Path(video_name).stem
    path = find_chapter_file(stem, CHAPTER_DIR)
    if path:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
            return data if isinstance(data, list) else data.get("chapters", [])
    return []


# ═══════════════════════════════════════════════════════════════════
#  Pipeline helpers
# ═══════════════════════════════════════════════════════════════════
def log_msg(job: dict, text: str) -> None:
    job["logs"].append({"t": time.strftime("%H:%M:%S"), "msg": text})
    job["step"] = text


_FILLER = re.compile(
    r"\b(?:um+|uh+|like|you know|so|actually|basically|right)\b", re.I
)

def clean_text(text: str) -> str:
    return re.sub(r"\s+", " ", _FILLER.sub("", text)).strip()


# ── Semantic segmentation (SBERT cosine-similarity boundary detection) ──
def chunk_segments_semantic(segments: list, sbert_model, min_duration: float = 30.0) -> list:
    import numpy as np

    if not segments:
        return []
    if len(segments) == 1:
        s = segments[0]
        return [{"start": s["start"], "end": s["end"], "text": s["text"].strip()}]

    texts = [s["text"].strip() or " " for s in segments]
    embeddings = sbert_model.encode(
        texts, batch_size=32, show_progress_bar=False, normalize_embeddings=True
    )

    # Cosine similarities between adjacent segments (dot product on normalised vecs)
    sims = [float(embeddings[i] @ embeddings[i + 1]) for i in range(len(embeddings) - 1)]

    arr = np.array(sims)
    threshold = float(np.mean(arr) - 0.5 * np.std(arr))

    # Boundary candidates: below-threshold local minima
    breakpoints = []
    for i, s in enumerate(sims):
        if s < threshold:
            left_ok  = (i == 0             or s <= sims[i - 1])
            right_ok = (i == len(sims) - 1 or s <= sims[i + 1])
            if left_ok or right_ok:
                breakpoints.append(i + 1)

    def make_chunk(group):
        return {
            "start": group[0]["start"],
            "end":   group[-1]["end"],
            "text":  " ".join(sg["text"].strip() for sg in group),
        }

    chunks, prev = [], 0
    for bp in breakpoints:
        group = segments[prev:bp]
        if group:
            c = make_chunk(group)
            if c["end"] - c["start"] < min_duration and chunks:
                chunks[-1]["end"]   = c["end"]
                chunks[-1]["text"] += " " + c["text"]
            else:
                chunks.append(c)
        prev = bp

    group = segments[prev:]
    if group:
        c = make_chunk(group)
        if c["end"] - c["start"] < min_duration and chunks:
            chunks[-1]["end"]   = c["end"]
            chunks[-1]["text"] += " " + c["text"]
        else:
            chunks.append(c)

    return chunks


# ── Word-count cap: split chunks that exceed MAX_CHAPTER_WORDS ──────────
def split_long_chunks(chunks: list, segments: list, max_words: int = 300) -> list:
    """Split any chunk whose word count exceeds *max_words*.

    Re-maps each oversized chunk back to the original Whisper segments and
    accumulates segments into sub-chunks of ~max_words each.  Chunks that
    are already within the limit are passed through unchanged.
    """
    result = []
    for ch in chunks:
        if len(ch["text"].split()) <= max_words:
            result.append(ch)
            continue

        # Find the Whisper segments that fall within this chunk's time range
        ch_segs = [
            s for s in segments
            if s["start"] >= ch["start"] - 0.5 and s["end"] <= ch["end"] + 0.5
        ]
        if not ch_segs:
            result.append(ch)
            continue

        # Accumulate segments until ~max_words, then start a new sub-chunk
        buf_segs = []
        buf_words = 0
        for seg in ch_segs:
            seg_words = len(seg["text"].strip().split())
            if buf_words + seg_words > max_words and buf_segs:
                result.append({
                    "start": buf_segs[0]["start"],
                    "end":   buf_segs[-1]["end"],
                    "text":  " ".join(s["text"].strip() for s in buf_segs),
                })
                buf_segs = [seg]
                buf_words = seg_words
            else:
                buf_segs.append(seg)
                buf_words += seg_words

        # Flush remaining segments
        if buf_segs:
            result.append({
                "start": buf_segs[0]["start"],
                "end":   buf_segs[-1]["end"],
                "text":  " ".join(s["text"].strip() for s in buf_segs),
            })

    return result


# ── Visual feature extraction (ResNet18, CPU-optimised) ──────────────────
class VisualFeatureExtractor:
    def __init__(self, use_lightweight: bool = True):
        import torch
        import torch.nn as nn
        from torchvision import transforms, models

        if use_lightweight:
            base = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        else:
            base = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

        self.model = nn.Sequential(*list(base.children())[:-1])
        self.model.eval()

        self.preprocess = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225],
            ),
        ])

    def extract_from_video(self, video_path: str, sample_rate: int = 60, job: dict = None):
        import cv2
        import numpy as np
        import torch

        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            raise ValueError(f"Cannot open video: {video_path}")

        fps          = cap.get(cv2.CAP_PROP_FPS) or 25.0
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        frames_per_sample = max(1, int(fps * sample_rate))

        features, timestamps = [], []
        frame_count = 0

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            if frame_count % frames_per_sample == 0:
                current_time = frame_count / fps
                frame_rgb    = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                tensor       = self.preprocess(frame_rgb).unsqueeze(0)
                with torch.no_grad():
                    feat = self.model(tensor).squeeze().numpy()
                features.append(feat)
                timestamps.append(current_time)
                if job and len(features) % 10 == 0:
                    log_msg(job, f"[4/6] Visual features: {len(features)} frames sampled ({current_time:.0f}s)")
            frame_count += 1

        cap.release()
        return np.array(features), np.array(timestamps)


def align_visual_features(chunks: list, visual_features, visual_timestamps) -> list:
    import numpy as np

    aligned = []
    for ch in chunks:
        start, end = ch["start"], ch["end"]
        mask = (visual_timestamps >= start) & (visual_timestamps <= end)
        matching = visual_features[mask]
        if len(matching) > 0:
            avg = np.mean(matching, axis=0)
        else:
            nearest_idx = int(np.argmin(np.abs(visual_timestamps - start)))
            avg = visual_features[nearest_idx]
        aligned.append({**ch, "visual_feature": avg})
    return aligned


# ── LLM chapter generation (with robust retry + JSON parsing) ───────────
def generate_chapter_json(chapter_feature: dict, model, tokenizer, max_retries: int = 3) -> dict:
    import torch

    prompt = (
        "Generate a chapter summary JSON object for this video segment.\n\n"
        "Segment Information:\n"
        f"- Time Range: {chapter_feature['start_time']} to {chapter_feature['end_time']}\n"
        f"- Duration: {chapter_feature['duration_seconds']:.1f} seconds\n"
        f"- Keywords: {', '.join(chapter_feature.get('keywords', [])[:5]) or 'N/A'}\n"
        f"- Transcript: {chapter_feature['text'][:500]}\n\n"
        "Return ONLY a single valid JSON object with exactly these fields:\n"
        "{\n"
        f'  "ID": "{chapter_feature["chapter_id"] + 1}",\n'
        '  "chapter": "<descriptive title, 5-8 words>",\n'
        f'  "start_time": "{chapter_feature["start_time"]}",\n'
        f'  "end_time": "{chapter_feature["end_time"]}",\n'
        '  "description": "<1-2 sentence summary>"\n'
        "}\n\n"
        "Rules:\n"
        "- Output ONLY the JSON object, nothing else\n"
        "- No markdown, no code fences, no explanation\n"
        "- Keep description under 50 words\n"
        f'- ID must be the string "{chapter_feature["chapter_id"] + 1}"'
    )

    for attempt in range(max_retries):
        generated = ""
        try:
            messages  = [{"role": "user", "content": prompt}]
            formatted = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
            inputs = tokenizer(formatted, return_tensors="pt")

            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=300,
                    temperature=0.1,
                    do_sample=True,
                    top_p=0.9,
                    repetition_penalty=1.1,
                    pad_token_id=tokenizer.eos_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                )

            generated = tokenizer.decode(
                outputs[0][inputs["input_ids"].shape[-1]:],
                skip_special_tokens=True,
            ).strip()

            # Strip markdown fences
            generated = re.sub(r"```(?:json)?", "", generated).strip()

            # Extract the FIRST complete JSON object (brace counting)
            brace_count = 0
            start_idx = end_idx = None
            for i, ch in enumerate(generated):
                if ch == "{":
                    if start_idx is None:
                        start_idx = i
                    brace_count += 1
                elif ch == "}":
                    brace_count -= 1
                    if brace_count == 0 and start_idx is not None:
                        end_idx = i + 1
                        break

            if start_idx is None or end_idx is None:
                raise ValueError("No complete JSON object found in output")

            json_str = generated[start_idx:end_idx]

            # Fix common model quirks
            json_str = re.sub(r",\s*}", "}", json_str)           # trailing commas
            json_str = re.sub(r":\s*True\b",  ': "True"',  json_str)
            json_str = re.sub(r":\s*False\b", ': "False"', json_str)

            result = json.loads(json_str)

            # Normalise types
            result["ID"] = str(result.get("ID", chapter_feature["chapter_id"] + 1))

            # Enforce correct timestamps (model sometimes hallucinates these)
            result["start_time"] = chapter_feature["start_time"]
            result["end_time"]   = chapter_feature["end_time"]

            required = ["ID", "chapter", "start_time", "end_time", "description"]
            if all(f in result for f in required):
                return result

        except json.JSONDecodeError:
            pass
        except Exception:
            pass

    # Fallback
    return {
        "ID":          str(chapter_feature["chapter_id"] + 1),
        "chapter":     f"Chapter {chapter_feature['chapter_id'] + 1}",
        "start_time":  chapter_feature["start_time"],
        "end_time":    chapter_feature["end_time"],
        "description": chapter_feature["text"][:200].strip(),
    }


def classify_high_level(chapter: dict, model, tokenizer) -> str:
    import torch

    title_desc = (chapter.get("chapter", "") + " " + chapter.get("description", "")).lower()
    if any(k in title_desc for k in LOW_VALUE_KEYWORDS):
        return "False"

    prompt = (
        "Is this lecture chapter a key technical topic (not welcome/recap/tutorial/admin/Q&A)?\n"
        f"Title: {chapter.get('chapter')}\n"
        f"Description: {chapter.get('description')}\n"
        "Answer only Yes or No."
    )
    try:
        inp = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
        with torch.no_grad():
            out = model.generate(
                **inp,
                max_new_tokens=8,
                temperature=0.0,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        raw = tokenizer.decode(out[0], skip_special_tokens=True)
        if re.search(r"\b(yes|true)\b", raw, re.IGNORECASE):
            return "True"
    except Exception:
        pass
    return "False"


# ═══════════════════════════════════════════════════════════════════
#  Background pipeline runner  (6 steps)
# ═══════════════════════════════════════════════════════════════════
def run_pipeline(job_id: str, video_path: str, video_name: str):
    job  = jobs[job_id]
    stem = Path(video_name).stem
    audio_path      = os.path.join(TRANSCRIPT_FOLDER, f"{stem}.wav")
    transcript_path = os.path.join(TRANSCRIPT_FOLDER, f"{stem}_transcript.txt")

    try:
        # ── 1  Audio extraction ──────────────────────────────────────
        job["progress"] = "1 / 6"
        log_msg(job, "[1/6] Starting audio extraction via FFmpeg ...")
        res = subprocess.run(
            ["ffmpeg", "-y", "-i", video_path,
             "-ar", "16000", "-ac", "1", "-f", "wav", audio_path],
            capture_output=True, text=True, timeout=600,
        )
        if res.returncode != 0:
            raise RuntimeError(f"FFmpeg error:\n{res.stderr[:500]}")
        log_msg(job, f"[1/6] Audio extracted -> {os.path.basename(audio_path)}")

        # ── 2  Whisper transcription ─────────────────────────────────
        job["progress"] = "2 / 6"
        log_msg(job, "[2/6] Loading Whisper base model ...")
        import whisper as _whisper
        wmodel = _whisper.load_model("base")
        if not (os.path.exists(transcript_path)):
            log_msg(job, "[2/6] Transcribing audio (this may take a while) ...")
            result   = wmodel.transcribe(audio_path, language="en")
            segments = result.get("segments", [])
            if not segments:
                raise RuntimeError("Whisper produced no segments - is the audio silent?")
            duration = segments[-1]["end"] if segments else 0
            log_msg(job, f"[2/6] Transcription done - {len(segments)} segments, {fmt_ts(duration)} total")
            with open(transcript_path, "w", encoding="utf-8") as f:
                for seg in segments:
                    f.write(f"[{fmt_ts(seg['start'])} --> {fmt_ts(seg['end'])}] {seg['text'].strip()}\n")
            log_msg(job, f"[2/6] Transcript saved -> {os.path.basename(transcript_path)}")

        else:
            log_msg(job, f"[2/6] Transcript already exists - loading from {os.path.basename(transcript_path)} ...")
            segments = []
            with open(transcript_path, "r", encoding="utf-8") as f:
                for line in f:
                    match = re.match(r"\[(\d{2}:\d{2}:\d{2}) --> (\d{2}:\d{2}:\d{2})\] (.+)", line)
                    if match:
                        start, end, text = match.groups()
                        segments.append({
                            "start": parse_time(start),
                            "end": parse_time(end),
                            "text": text.strip(),
                        })
            log_msg(job, f"[2/6] Loaded {len(segments)} segments from existing transcript")


        # ── 3  Semantic segmentation + NLP features ──────────────────
        job["progress"] = "3 / 6"
        log_msg(job, "[3/6] Loading SBERT model (all-MiniLM-L6-v2) ...")
        from sentence_transformers import SentenceTransformer
        from keybert import KeyBERT
        import numpy as np

        sbert    = SentenceTransformer("all-MiniLM-L6-v2")
        kw_model = KeyBERT(model=sbert)

        log_msg(job, f"[3/6] Running semantic segmentation on {len(segments)} transcript segments ...")
        chunks = chunk_segments_semantic(segments, sbert)
        log_msg(job, f"[3/6] Segmentation done - {len(chunks)} semantic chunks identified")

        # Enforce max word count: split chunks exceeding MAX_CHAPTER_WORDS
        pre_split = len(chunks)
        chunks = split_long_chunks(chunks, segments, max_words=MAX_CHAPTER_WORDS)
        if len(chunks) != pre_split:
            log_msg(job, f"[3/6] Word-count cap ({MAX_CHAPTER_WORDS} words): {pre_split} -> {len(chunks)} chunks")
        else:
            log_msg(job, f"[3/6] Word-count cap ({MAX_CHAPTER_WORDS} words): all chunks within limit")

        log_msg(job, f"[3/6] Generating SBERT text embeddings for {len(chunks)} chunks ...")
        chunk_texts  = [clean_text(ch["text"]) or " " for ch in chunks]
        text_embeds  = sbert.encode(
            chunk_texts, batch_size=8, show_progress_bar=False, normalize_embeddings=True
        )
        log_msg(job, f"[3/6] Text embeddings done - dim={text_embeds.shape[1]}")

        features = []
        for i, ch in enumerate(chunks):
            txt = clean_text(ch["text"])
            if not txt:
                continue
            kws = [w for w, _ in kw_model.extract_keywords(txt, top_n=5, stop_words="english")]
            features.append({
                "chapter_id":       i,
                "start_time":       fmt_ts(ch["start"]),
                "end_time":         fmt_ts(ch["end"]),
                "start_seconds":    float(ch["start"]),
                "end_seconds":      float(ch["end"]),
                "duration_seconds": float(ch["end"] - ch["start"]),
                "text":             txt,
                "word_count":       len(txt.split()),
                "keywords":         kws,
                "text_embedding":   text_embeds[i].tolist(),
            })
            log_msg(job, f"[3/6] KeyBERT chunk {i + 1}/{len(chunks)}: {', '.join(kws[:3])}")
            job["progress"] = "3 / 6"

        if not features:
            raise RuntimeError("No text features could be extracted from the transcript.")
        log_msg(job, f"[3/6] NLP features done - {len(features)} chunks ready")

        # ── 4  Visual feature extraction (ResNet18) ──────────────────
        job["progress"] = "4 / 6"
        log_msg(job, f"[4/6] Loading ResNet18 for visual feature extraction ...")
        extractor = VisualFeatureExtractor(use_lightweight=True)
        log_msg(job, f"[4/6] Extracting visual features (1 frame per {FRAME_SAMPLE_RATE}s) ...")
        vis_feats, vis_times = extractor.extract_from_video(
            video_path, sample_rate=FRAME_SAMPLE_RATE, job=job
        )
        log_msg(job, f"[4/6] Extracted {len(vis_feats)} visual frames - dim={vis_feats.shape[1]}")

        log_msg(job, "[4/6] Aligning visual features with semantic chunks ...")
        # Build chunk dicts from features for alignment
        feat_chunks = [
            {"start": f["start_seconds"], "end": f["end_seconds"], "text": f["text"]}
            for f in features
        ]
        aligned = align_visual_features(feat_chunks, vis_feats, vis_times)
        for i, feat in enumerate(features):
            feat["visual_feature"] = aligned[i]["visual_feature"].tolist()
            feat["visual_feature_dim"] = len(aligned[i]["visual_feature"])
        log_msg(job, f"[4/6] Visual alignment done - {len(features)} chunks enriched")

        # ── 5  LLM chapter generation + key/skip classification ──────
        job["progress"] = "5 / 6"
        log_msg(job, "[5/6] Loading Qwen2.5-1.5B-Instruct (CPU, float32) ...")
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer

        llm_name  = "Qwen/Qwen2.5-1.5B-Instruct"
        tokenizer = AutoTokenizer.from_pretrained(llm_name, trust_remote_code=True)
        llm       = AutoModelForCausalLM.from_pretrained(
            llm_name,
            dtype=torch.float32,         # torch_dtype deprecated in transformers 5.x
            trust_remote_code=True,
            low_cpu_mem_usage=True,
        )
        llm.eval()
        log_msg(job, "[5/6] LLM loaded - generating chapters ...")

        chapters = []
        for i, feat in enumerate(features):
            job["progress"] = f"5 / 6  ({i + 1}/{len(features)})"
            log_msg(job, f"[5/6] Generating chapter {i + 1}/{len(features)} [{feat['start_time']}] ...")
            ch_json = generate_chapter_json(feat, llm, tokenizer)
            ch_json["keywords"] = feat["keywords"]
            log_msg(job, f"[5/6]   -> \"{ch_json['chapter']}\"")
            chapters.append(ch_json)

        log_msg(job, f"[5/6] Classifying {len(chapters)} chapters as key/skip ...")
        for i, ch in enumerate(chapters):
            ch["key"] = classify_high_level(ch, llm, tokenizer)
            log_msg(job, f"[5/6]   [{i + 1}/{len(chapters)}] {ch['chapter']} -> {'KEY' if ch['key'] == 'True' else 'skip'}")
            job["progress"] = f"5 / 6  ({i + 1}/{len(chapters)})"

        # ── 6  Save ──────────────────────────────────────────────────
        job["progress"] = "6 / 6"
        log_msg(job, "[6/6] Saving chapters with tumbling window segmentation ...")
        out_path = os.path.join(CHAPTER_DIR, f"{stem}-tm-chapters.json")
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(chapters, f, indent=2, ensure_ascii=False)
        log_msg(job, f"[6/6] Done! Chapters saved to {out_path}")

        job.update(status="done", step="Pipeline complete!", progress="6 / 6")

    except Exception as exc:
        err = f"{type(exc).__name__}: {exc}"
        log_msg(job, f"ERROR: {err}")
        job.update(status="error", error=err, step="Pipeline failed")
        traceback.print_exc()


# ═══════════════════════════════════════════════════════════════════
#  Flask routes
# ═══════════════════════════════════════════════════════════════════
@app.route("/")
def index():
    videos = get_available_videos()
    return render_template(
        "UI-template.html",
        chapters_json="[]",
        total_duration=0,
        video_path="",
        available_videos=videos,
        current_video=None,
        has_chapters=False,
    )


@app.route("/video/<video_name>")
def view_video(video_name):
    vfile = os.path.join(UPLOAD_FOLDER, video_name)
    if not os.path.exists(vfile):
        return redirect("/")

    chapters = load_chapters(video_name)
    enriched = enrich_chapters(chapters) if chapters else []
    total    = max((c["end_sec"] for c in enriched), default=0)
    videos   = get_available_videos()

    return render_template(
        "UI-template.html",
        chapters_json=json.dumps(enriched),
        total_duration=total,
        video_path=url_for("static", filename=f"videos/{video_name}"),
        available_videos=videos,
        current_video=video_name,
        has_chapters=len(chapters) > 0,
    )


@app.route("/upload", methods=["POST"])
def upload():
    if "video" not in request.files:
        return jsonify(status="error", message="No file provided"), 400
    f = request.files["video"]
    if not f.filename:
        return jsonify(status="error", message="Empty filename"), 400
    ext = Path(f.filename).suffix.lower()
    if ext not in ALLOWED_EXT:
        return jsonify(status="error", message=f"Unsupported format: {ext}"), 400

    safe = f.filename.replace(" ", "_")
    dest = os.path.join(UPLOAD_FOLDER, safe)
    f.save(dest)

    jid = uuid.uuid4().hex[:8]
    jobs[jid] = dict(status="running", step="Uploaded - queuing pipeline ...", progress="0 / 6", error=None, logs=[])
    threading.Thread(target=run_pipeline, args=(jid, dest, safe), daemon=True).start()
    return jsonify(status="ok", job_id=jid, video_name=safe)


@app.route("/process/<video_name>", methods=["POST"])
def process_existing(video_name):
    vpath = os.path.join(UPLOAD_FOLDER, video_name)
    if not os.path.exists(vpath):
        return jsonify(status="error", message="Video not found"), 404

    jid = uuid.uuid4().hex[:8]
    jobs[jid] = dict(status="running", step="Starting pipeline ...", progress="0 / 6", error=None, logs=[])
    threading.Thread(target=run_pipeline, args=(jid, vpath, video_name), daemon=True).start()
    return jsonify(status="ok", job_id=jid, video_name=video_name)


@app.route("/api/job/<job_id>")
def job_status(job_id):
    job = jobs.get(job_id)
    if not job:
        return jsonify(status="error", error="Unknown job"), 404
    return jsonify(job)


# ═══════════════════════════════════════════════════════════════════
if __name__ == "__main__":
    print("AutoChapter running at http://localhost:3000")
    app.run(debug=True, use_reloader=False)
